**warning:**
priori only works at google colab. it requires python3.5 which is hard to prepare at venv or conda.

### **Ml Text**

using text csv data for ml



**index:**

1 nltk

2 apriori


**description:**

nlktk - ml predicts comment is positive/negative.

apriori - apriori helps to predict another item to buy when customer select an item.

**1 nlktk**

**restaurant review**

ml predicts comment is positive/negative.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import pickle
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score,confusion_matrix
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from google.colab import drive
drive.mount('/content/drive')
sc=StandardScaler()
ps=PorterStemmer()
f1=pd.read_csv('/content/drive/MyDrive/ml/asset/restaurant_reviews.txt', delimiter='\t',quoting=3)

# 0 Create nltk_data folder at your local(User folder level)
nltk.download('all')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [ ]:
# 1 Create trimed dataset x=comment, y=positive_negative
txt=[]
for i in range(0, 1000):
  x1=re.sub('[^a-zA-Z]', ' ',f1['Review'][i])
  x1=x1.lower().split()
  x2=[ps.stem(word) for word in x1 if not word in set(stopwords.words('english'))]
  x2=' '.join(x2)
  txt.append(x2)

vectorizer=TfidfVectorizer(max_features=1500,min_df=3,max_df=0.6)
x_f1=vectorizer.fit_transform(txt).toarray()
y_f1=f1.iloc[:,1].values
# print(x_f1[0])
# print(y_f1)

#find deviation with stdev of 1* *avr btw next item
x_80,x_20,y_80,y_20=train_test_split(x_f1,y_f1,test_size=0.20,random_state=0)

# 3.1 Find purchase rate per customer with KNeighbor Classification
classifier=KNeighborsClassifier(n_neighbors=5,metric='minkowski',p=2)
classifier.fit(x_80,y_80)
y_pred=classifier.predict(x_20)
y_prob=classifier.predict_proba(x_20)[:,1]

# # 3.2 Find accuracy level by confusion matrix result(1001 items):suppposed to be 0.58
cm=confusion_matrix(y_20,y_pred)

# # ⭐︎3.3.1 way1 estimate purchase rate mannuary
sample1=['Good batting by England;']
sample1=vectorizer.transform(sample1).toarray()
sentiment = classifier.predict(sample1)
print('\ngiven text: Good batting by England\nml predict result:',sentiment,'\n*0-negative,1-positive') # supporsed to be 1, but I got 0
sample2=['The chips and salsa were really good.']
sample2=vectorizer.transform(sample2).toarray()
sentiment2=classifier.predict(sample2)
print('\ngiven text: The chips and salsa were really good.\nml predict result:',sentiment2,'\n*0-negative,1-positive') # supporsed to be 1, but I got 0

# ------OPTION----------
# Check1 property of the data
# print('\n• Check1 property of the data:\n',f1.info(),f1.head())
# Check2 created items
# print('\n• Check2 created items:\nitem1:',txt[0],'\nitem2:',txt[6],'\nitem3:',txt[12])
# Check3 acculacy level
# print('\n• Check3 acculacy level:\n',accuracy_score(y_20,y_pred))
# Check4
# print('\n• Check4 Find accuracy level by confusion matrix:\n',cm,'\nfirst row: T ok, F wrong, second row:F wrong, T ok (115 items are correct out of 200)')#[[77 20][54 49]]



given text: Good batting by England
ml predict result: [1] 
*0-negative,1-positive

given text: The chips and salsa were really good.
ml predict result: [1] 
*0-negative,1-positive


**2 apriori**

**customer also buy this items**

apriori helps to predict another item to buy when customer select an item.

In [ ]:
!pip install apyori

  Preparing metadata (setup.py) ... done
  Created wheel for apyori: filename=apyori-1.1.2-py3-none-any.whl size=5954 sha256=c4dfbac481919313ce314116a2a9cfca05f8051de2f27075c9cd5bdb7af6f56a
  Stored in directory: /root/.cache/pip/wheels/7f/49/e3/42c73b19a264de37129fadaa0c52f26cf50e87de08fb9804af
Successfully built apyori


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from apyori import apriori
from google.colab import drive
dataset = pd.read_csv('/content/drive/MyDrive/ml/asset/grocery_list.csv', header = None)
transactions = []
for i in range(0, 7501):
  transactions.append([str(dataset.values[i,j]) for j in range(0, 20)])

rules=apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2, max_length = 2)
results=list(rules)
print(results)
# detailed:
# def inspect(results):
#     lhs         = [tuple(result[2][0][0])[0] for result in results]
#     rhs         = [tuple(result[2][0][1])[0] for result in results]
#     supports    = [result[1] for result in results]
#     confidences = [result[2][0][2] for result in results]
#     lifts       = [result[2][0][3] for result in results]
#     return list(zip(lhs, rhs, supports, confidences, lifts))
# resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])
# resultsinDataFrame
# resultsinDataFrame.nlargest(n = 10, columns = 'Lift')

# simpler:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    return list(zip(lhs, rhs, supports))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Product 1', 'Product 2', 'Support'])
resultsinDataFrame.nlargest(n = 10, columns = 'Support')


[RelationRecord(items=frozenset({'light cream', 'chicken'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]), RelationRecord(items=frozenset({'mushroom cream sauce', 'escalope'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]), RelationRecord(items=frozenset({'pasta', 'escalope'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]), RelationRecord(items=frozenset({'fromage blanc', 'honey'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0.24

,Product 1,Product 2,Support
4,herb & pepper,ground beef,0.015998
7,whole wheat pasta,olive oil,0.007999
2,pasta,escalope,0.005866
1,mushroom cream sauce,escalope,0.005733
5,tomato sauce,ground beef,0.005333
8,pasta,shrimp,0.005066
0,light cream,chicken,0.004533
3,fromage blanc,honey,0.003333
6,light cream,olive oil,0.003200


Result you can see from table above: When customer buy herb&pepper, the person likely to get ground beef.